In [1]:
from py_module.metadata.connection.ConnectionRegistry import PostgresConnection as pgconn
from py_module.metadata.render_jinja.RenderRegistry import ChangeDataCaptureExtraction as cdc, RetrieveSchema as rs, SchemaSource as ss 
from py_module.metadata.connection.StorageBaseConnection import StorageConn as storage
from py_module.metadata.datatype_conversion.avro import DataTypeConverter as dtc
from sqlalchemy import text
import json
from fastavro import writer, parse_schema
from datetime import datetime as dt, timedelta


# SETUP (connection + get template)

In [2]:
params = {
    "database":"postgres",
    "db": "meteo",
    "host": "localhost",
    "port": 5433,
    "user": "meteo",
    "password": "meteo",
    "ssl_args": {}
} 

In [3]:
engine = pgconn.get_engine(**params)
conn = engine.connect()

In [4]:
cdc_extraction = cdc.render_jinja(database=params['database'])
retrieve_schema = rs.render_jinja(database=params['database'])
schema_source = ss.render_jinja()

# Render template with appropriate values

### Retrieve schema table -- this is useful for next steps

In [5]:
table_schema = 'public'
table_name = 'fct_meteo'
rendered_schema = retrieve_schema.render(
                                table_schema=table_schema,
                                table_name=table_name
                            )

In [6]:
it_schema = conn.execute(text(rendered_schema))
converter = dtc()  # your DataTypeConverter instance, optionally pass defaults

dbt_columns = list()
avro_columns = list()
cdc_columns = list()

for i in it_schema:
    col_name = i[0]
    db_type = i[1]
    precision = i[2]
    scale = i[3]

    avro_type = converter.source_to_avro(params['database'], db_type, numeric_precision=precision, numeric_scale=scale)
    bq_type = converter.source_to_bigquery(params['database'], db_type)

    # setup for cdc model injection
    cdc_columns.append(col_name)

    # setup the columns for avro injection with default
    avro_field = converter.generate_avro_field(col_name, avro_type)
    avro_columns.append(avro_field)

    # setup the columns for dbt source
    dbt_columns.append({col_name: bq_type})


In [7]:

rendered_sources = schema_source.render(
    schema_name = table_schema,
    table_name = table_name,
    cols = dbt_columns,
    database = params['database'],
    staging_dataset = 'staging',
    istance_name = params['db'],
    bucket_name = 'postgres__d-meteo-db',
    version = 'v1'
    )

### define custom CDC extraction query 

In [8]:
rendered_cdc = cdc_extraction.render(
                                columns = cdc_columns,
                                schema_name = table_schema, 
                                table_name = table_name, 
                                delta = False, 
                                # delta_column = delta_column, 
                                # delta_timestamp = delta_timestamp, 
                                # where_conditions = where_conditions
                            )

In [9]:
avro_schema = {
    "name": f"{table_schema}_{table_name}__record",
    "type": "record",
    "fields": avro_columns,
}
parsed_schema = parse_schema(avro_schema)

In [24]:
today = (dt.today() + timedelta(days=-2)).strftime("%Y-%m-%d")
now = dt.now().strftime("%d%m%Y%H%M%S")
gcs = storage()
blob = gcs.define_blob(
    bucket_name='postgres__d-meteo-db',
    blob_name=f'{table_schema}/{table_name}/ingestion_date={today}/chunk__{now}.avro'
)


Authenticated to storage (using default credentials)


In [36]:
cdc_exec = conn.execution_options(stream_results=True).execute(text(rendered_cdc))
cols = cdc_exec.keys()

In [ ]:
import time
import logging
from fastavro import writer
from decimal import Decimal, ROUND_DOWN

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

MAX_FILE_SIZE = 400 * 1024 * 1024  # 400 MB from source (around 5 time smaller on target)
decimal_fields = {}
for field in parsed_schema['fields']:
    f_type = field['type']
    if isinstance(f_type, dict) and f_type.get('logicalType') == 'decimal':
        decimal_fields[field['name']] = f_type['scale']
    elif isinstance(f_type, list):
        for t in f_type:
            if isinstance(t, dict) and t.get('logicalType') == 'decimal':
                decimal_fields[field['name']] = t['scale']

file_counter = 1

def fix_decimal(value, scale):
    quantize_map = {i: Decimal('0.' + '0' * (i-1) + '1') for i in range(1, 21)}
    if value is None:
        return None
    return Decimal(value).quantize(quantize_map[scale], rounding=ROUND_DOWN)

def record_generator():
    for row in cdc_exec:
        record = dict(zip(cols, row))
        for field, scale in decimal_fields.items():
            if field in record and record[field] is not None:
                record[field] = fix_decimal(record[field], scale)
        yield record

def upload_avro_chunks():
    global file_counter
    record_iter = record_generator()

    total_records_written = 0
    file_records = []

    while True:
        chunk_records = []
        chunk_size = 0
        start_time = time.time()

        # Accumulate records for this chunk
        try:
            while True:
                record = next(record_iter)
                chunk_records.append(record)

                # Rough byte size estimate for chunking
                chunk_size += len(str(record).encode('utf-8'))
                if chunk_size >= MAX_FILE_SIZE:
                    break

        except StopIteration:
            if not chunk_records:
                break  # no more records

        # Define blob and write Avro file
        blob_name = f'{table_schema}/{table_name}/ingestion_date={today}/chunk__{now}__part{file_counter}.avro'
        blob = gcs.define_blob(bucket_name='postgres__d-meteo-db', blob_name=blob_name)

        with blob.open('wb', ignore_flush=True) as f:
            writer(f, parsed_schema, chunk_records)

        records_written = len(chunk_records)
        total_records_written += records_written
        file_records.append(records_written)

        elapsed_time = time.time() - start_time
        throughput = (chunk_size / elapsed_time / (1024*1024)) if elapsed_time > 0 else 0
        logging.info(f"Chunk {file_counter}: {records_written} records, "
                     f"{chunk_size / (1024*1024):.2f} MB, "
                     f"{throughput:.2f} MB/s")

        file_counter += 1

    logging.info(f"Total records written: {total_records_written}")
    logging.info(f"Records per file: {file_records}")


In [ ]:
def record_generator():
    for row in cdc_exec:
        record = dict(zip(cols, row))
        
        # Process decimal fields dynamically
        for field in parsed_schema['fields']:
            f_name = field['name']
            f_type = field['type']
            
            # Handle union types (nullable fields)
            types = f_type if isinstance(f_type, list) else [f_type]
            
            for t in types:
                if isinstance(t, dict) and t.get('logicalType') == 'decimal':
                    if f_name in record and record[f_name] is not None:
                        record[f_name] = fix_decimal(record[f_name], t['scale'])
        yield record

In [38]:
upload_avro_chunks()

2025-10-19 14:35:34,896 [INFO] Chunk 1: 129309 records, 100.00 MB, 1.12 MB/s
2025-10-19 14:36:22,512 [INFO] Chunk 2: 60051 records, 46.55 MB, 0.98 MB/s
2025-10-19 14:36:22,575 [INFO] Total records written: 189360
2025-10-19 14:36:22,576 [INFO] Records per file: [129309, 60051]


In [ ]:
from decimal import Decimal, ROUND_DOWN
# Map decimal fields to their scale from schema
decimal_fields = {}
for field in parsed_schema['fields']:
    f_type = field['type']
    if isinstance(f_type, dict) and f_type.get('logicalType') == 'decimal':
        decimal_fields[field['name']] = f_type['scale']
    # handle nullable decimals
    elif isinstance(f_type, list):
        for t in f_type:
            if isinstance(t, dict) and t.get('logicalType') == 'decimal':
                decimal_fields[field['name']] = t['scale']


def fix_decimal(value, scale):
    """Ensure value is a Decimal with correct scale."""
    if value is None:
        return None
    # convert float or string to Decimal
    d = Decimal(value)
    quantize_str = '0.' + '0' * (scale - 1) + '1'  # e.g., "0.01" for scale 2
    return d.quantize(Decimal(quantize_str), rounding=ROUND_DOWN)

def record_generator():
    for row in cdc_exec:
        record = dict(zip(cols, row))
        for field, scale in decimal_fields.items():
            if field in record and record[field] is not None:
                record[field] = fix_decimal(record[field], scale)
        yield record


In [ ]:
with blob.open('wb', ignore_flush=True) as f:
    with writer(f, parsed_schema) as w:
        for record in record_generator():
            w.write(record)

In [ ]:

    # Stream rows and write one-by-one
    for row in result:
        record = {}
        for col_name, value in zip(cols, row):
            # If your schema expects non-nullable fields, handle nulls here if needed
            record[col_name] = value
        avro_writer.write(record)

[{'name': 'observation_pk', 'type': 'text'},
 {'name': 'region', 'type': 'text'},
 {'name': 'province', 'type': 'text'},
 {'name': 'province_code', 'type': 'text'},
 {'name': 'city', 'type': 'text'},
 {'name': 'observation_dt', 'type': 'timestamp'},
 {'name': 'summary', 'type': 'text'},
 {'name': 'precip_intensity', 'type': 'numeric'},
 {'name': 'precip_accumulation', 'type': 'numeric'},
 {'name': 'precip_type', 'type': 'text'},
 {'name': 'temperature', 'type': 'numeric'},
 {'name': 'apparent_temperature', 'type': 'numeric'},
 {'name': 'dew_point', 'type': 'numeric'},
 {'name': 'pressure', 'type': 'numeric'},
 {'name': 'wind_speed', 'type': 'numeric'},
 {'name': 'wind_gust', 'type': 'numeric'},
 {'name': 'windb_earing', 'type': 'numeric'},
 {'name': 'cloud_cover', 'type': 'numeric'},
 {'name': 'snow_accumulation', 'type': 'numeric'},
 {'name': 'insert_timestamp', 'type': 'timestamp'},
 {'name': 'update_timestamp', 'type': 'timestamp'}]